# 07 · FastAPI Inference Service

Goal:

1. Package the validation champion, Logistic Regression, behind a FastAPI service.
2. Validate requests with Pydantic.
3. Load the model once when the application starts.
4. Expose `GET /health`, `GET /model-info`, and `POST /predict`.
5. Verify API predictions exactly match direct model predictions.
6. Save reusable API source files into Google Drive.

The model was trained on **19 engineered numeric features plus ticker**. Therefore the MVP API accepts those exact model-ready inputs. A later data pipeline/dashboard will calculate them automatically.

In [ ]:
!pip install -q \
    fastapi==0.141.1 \
    pydantic==2.13.5 \
    uvicorn==0.52.4 \
    httpx

In [ ]:
from google.colab import drive
from pathlib import Path

import json
import shutil
import sys
from importlib.metadata import version as package_version

import fastapi
import joblib
import numpy as np
import pandas as pd
import pydantic
import sklearn
import uvicorn

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ai-tech-market-risk")

API_DIR = PROJECT_ROOT / "api"
MODEL_DIR = PROJECT_ROOT / "models"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

API_DIR.mkdir(parents=True, exist_ok=True)

CHAMPION_MODEL_FILE = MODEL_DIR / "logistic_regression_candidate.joblib"
ML_DATA_FILE = PROCESSED_DATA_DIR / "ml_features.csv"

print("FastAPI:", fastapi.__version__)
print("Pydantic:", pydantic.__version__)
print("Uvicorn:", uvicorn.__version__)
print("scikit-learn:", sklearn.__version__)
print("API directory:", API_DIR)


## 1. Verify the champion artifact

In [ ]:
for required_file in [
    CHAMPION_MODEL_FILE,
    ML_DATA_FILE,
]:
    print(
        required_file.name,
        "exists:",
        required_file.exists(),
    )

    if not required_file.exists():
        raise FileNotFoundError(
            required_file
        )

## 2. Define the exact model inputs

In [ ]:
FEATURE_COLUMNS = [
    "Return_1D",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D",
    "Volume_Change_1D",
    "Relative_Volume_20D",
    "Price_vs_MA_5D",
    "Price_vs_MA_20D",
    "SPY_Return_1D",
    "QQQ_Return_1D",
    "SMH_Return_1D",
    "SPY_Return_5D",
    "QQQ_Return_5D",
    "SMH_Return_5D",
    "Excess_vs_QQQ_1D",
    "Excess_vs_SMH_1D",
]

STOCKS = [
    "NVDA",
    "AMD",
    "MSFT",
    "GOOGL",
    "META",
]

MODEL_COLUMNS = (
    FEATURE_COLUMNS + ["Ticker"]
)

print(
    "Numeric features:",
    len(FEATURE_COLUMNS),
)
print(
    "Total model inputs:",
    len(MODEL_COLUMNS),
)

In [ ]:
API_ARTIFACT_DIR = API_DIR / "artifacts"
API_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_FILE = API_ARTIFACT_DIR / "model_metadata.json"
METADATA = {
    "model_name": "LogisticRegression",
    "model_family": "Logistic Regression",
    "ticker_universe": STOCKS,
    "validation_roc_auc": 0.686896,
    "validation_pr_auc": 0.675456,
    "horizon_trading_days": 5,
    "move_threshold": 0.03,
}

METADATA_FILE.write_text(
    json.dumps(METADATA, indent=2),
    encoding="utf-8",
)

print("Created:", METADATA_FILE)

## 3. Write the FastAPI package

In [ ]:
SCHEMAS_SOURCE = 'from typing import Literal\n\nfrom pydantic import BaseModel, ConfigDict\n\n\nclass PredictionRequest(BaseModel):\n    model_config = ConfigDict(\n        extra="forbid",\n        allow_inf_nan=False,\n    )\n\n    Return_1D: float\n    Return_5D: float\n    Return_10D: float\n    Return_20D: float\n    Volatility_5D: float\n    Volatility_10D: float\n    Volatility_20D: float\n    Volume_Change_1D: float\n    Relative_Volume_20D: float\n    Price_vs_MA_5D: float\n    Price_vs_MA_20D: float\n    SPY_Return_1D: float\n    QQQ_Return_1D: float\n    SMH_Return_1D: float\n    SPY_Return_5D: float\n    QQQ_Return_5D: float\n    SMH_Return_5D: float\n    Excess_vs_QQQ_1D: float\n    Excess_vs_SMH_1D: float\n\n    Ticker: Literal[\n        "NVDA",\n        "AMD",\n        "MSFT",\n        "GOOGL",\n        "META",\n    ]\n\n\nclass PredictionResponse(BaseModel):\n    ticker: str\n    large_move_probability: float\n    predicted_large_move: bool\n    classification_threshold: float\n    model_name: str\n    horizon_trading_days: int\n    move_threshold: float\n\n\nclass HealthResponse(BaseModel):\n    status: str\n    model_loaded: bool\n\n\nclass ModelInfoResponse(BaseModel):\n    model_name: str\n    model_family: str\n    feature_count: int\n    ticker_universe: list[str]\n    horizon_trading_days: int\n    move_threshold: float\n    validation_roc_auc: float\n    validation_pr_auc: float\n'
SCHEMAS_FILE = API_DIR / 'schemas.py'
SCHEMAS_FILE.write_text(SCHEMAS_SOURCE, encoding='utf-8')
print('Created:', SCHEMAS_FILE)

In [ ]:
MODEL_SERVICE_SOURCE = 'from pathlib import Path\n\nimport joblib\nimport pandas as pd\n\n\nFEATURE_COLUMNS = [\n    "Return_1D",\n    "Return_5D",\n    "Return_10D",\n    "Return_20D",\n    "Volatility_5D",\n    "Volatility_10D",\n    "Volatility_20D",\n    "Volume_Change_1D",\n    "Relative_Volume_20D",\n    "Price_vs_MA_5D",\n    "Price_vs_MA_20D",\n    "SPY_Return_1D",\n    "QQQ_Return_1D",\n    "SMH_Return_1D",\n    "SPY_Return_5D",\n    "QQQ_Return_5D",\n    "SMH_Return_5D",\n    "Excess_vs_QQQ_1D",\n    "Excess_vs_SMH_1D",\n]\n\nMODEL_COLUMNS = FEATURE_COLUMNS + ["Ticker"]\n\nCLASSIFICATION_THRESHOLD = 0.5\nHORIZON_TRADING_DAYS = 5\nMOVE_THRESHOLD = 0.03\nMODEL_NAME = "LogisticRegression"\n\nMODEL_FILE = (\n    Path(__file__).resolve().parent\n    / "artifacts"\n    / "logistic_regression_candidate.joblib"\n)\n\nif not MODEL_FILE.exists():\n    raise FileNotFoundError(\n        f"Champion model not found: {MODEL_FILE}"\n    )\n\nmodel = joblib.load(MODEL_FILE)\n\n\ndef predict_large_move(payload: dict) -> dict:\n    row = pd.DataFrame(\n        [payload],\n        columns=MODEL_COLUMNS,\n    )\n\n    probability = float(\n        model.predict_proba(row)[0, 1]\n    )\n\n    predicted_large_move = (\n        probability >= CLASSIFICATION_THRESHOLD\n    )\n\n    return {\n        "ticker": payload["Ticker"],\n        "large_move_probability": probability,\n        "predicted_large_move": bool(\n            predicted_large_move\n        ),\n        "classification_threshold": (\n            CLASSIFICATION_THRESHOLD\n        ),\n        "model_name": MODEL_NAME,\n        "horizon_trading_days": (\n            HORIZON_TRADING_DAYS\n        ),\n        "move_threshold": MOVE_THRESHOLD,\n    }\n'
MODEL_SERVICE_FILE = API_DIR / 'model_service.py'
MODEL_SERVICE_FILE.write_text(MODEL_SERVICE_SOURCE, encoding='utf-8')
print('Created:', MODEL_SERVICE_FILE)

In [ ]:
APP_SOURCE = 'from fastapi import FastAPI\n\nfrom model_service import (\n    FEATURE_COLUMNS,\n    HORIZON_TRADING_DAYS,\n    MODEL_NAME,\n    MOVE_THRESHOLD,\n    model,\n    predict_large_move,\n)\nfrom schemas import (\n    HealthResponse,\n    ModelInfoResponse,\n    PredictionRequest,\n    PredictionResponse,\n)\n\nMODEL_METADATA_FILE = (\n    __import__("pathlib").Path(__file__).resolve().parent\n    / "artifacts"\n    / "model_metadata.json"\n)\n\nif not MODEL_METADATA_FILE.exists():\n    raise FileNotFoundError(\n        f"Model metadata not found: {MODEL_METADATA_FILE}"\n    )\n\nwith MODEL_METADATA_FILE.open("r", encoding="utf-8") as file:\n    MODEL_METADATA = __import__("json").load(file)\n\napp = FastAPI(\n    title="AI Tech Market Risk API",\n    version="0.1.0",\n    description=(\n        "Predicts the probability that a tracked "\n        "AI/technology stock experiences an "\n        "absolute move greater than 3% over the "\n        "next five trading days."\n    ),\n)\n\n\n@app.get("/health", response_model=HealthResponse)\ndef health():\n    return {\n        "status": "ok",\n        "model_loaded": model is not None,\n    }\n\n\n@app.get("/model-info", response_model=ModelInfoResponse)\ndef model_info():\n    return {\n        "model_name": MODEL_NAME,\n        "model_family": MODEL_METADATA["model_family"],\n        "feature_count": len(FEATURE_COLUMNS),\n        "ticker_universe": MODEL_METADATA["ticker_universe"],\n        "horizon_trading_days": HORIZON_TRADING_DAYS,\n        "move_threshold": MOVE_THRESHOLD,\n        "validation_roc_auc": MODEL_METADATA["validation_roc_auc"],\n        "validation_pr_auc": MODEL_METADATA["validation_pr_auc"],\n    }\n\n\n@app.post("/predict", response_model=PredictionResponse)\ndef predict(request: PredictionRequest):\n    return predict_large_move(request.model_dump())\n'
APP_FILE = API_DIR / "app.py"
APP_FILE.write_text(APP_SOURCE, encoding="utf-8")
print("Created:", APP_FILE)

## 4. Copy the frozen champion model into the API package

The API gets its own copy of the selected artifact so the service can later be containerized without depending on your whole notebook workspace.

In [ ]:
API_MODEL_FILE = (
    API_ARTIFACT_DIR
    / "logistic_regression_candidate.joblib"
)

shutil.copy2(
    CHAMPION_MODEL_FILE,
    API_MODEL_FILE,
)

print("Copied:", API_MODEL_FILE)
print("Size:", API_MODEL_FILE.stat().st_size)

## 5. Save deployment requirements

In [ ]:
REQUIREMENTS = f"""fastapi=={package_version("fastapi")}
pydantic=={package_version("pydantic")}
uvicorn=={package_version("uvicorn")}
pandas=={package_version("pandas")}
scikit-learn=={package_version("scikit-learn")}
joblib=={package_version("joblib")}
"""

REQUIREMENTS_FILE = API_DIR / "requirements.txt"
REQUIREMENTS_FILE.write_text(
    REQUIREMENTS,
    encoding="utf-8",
)

print(REQUIREMENTS_FILE.read_text())

## 6. Load the generated application

We test the actual source files we just created rather than defining a second copy of the API inside the notebook.

In [ ]:
api_path = str(API_DIR)

if api_path not in sys.path:
    sys.path.insert(
        0,
        api_path,
    )

for module_name in [
    "app",
    "schemas",
    "model_service",
]:
    sys.modules.pop(
        module_name,
        None,
    )

import app as api_app_module

app = api_app_module.app

print("FastAPI app loaded.")

## 7. Smoke-test `/health` and `/model-info`

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

health_response = client.get(
    "/health"
)

print(
    "Health status:",
    health_response.status_code,
)
print(
    health_response.json()
)

if health_response.status_code != 200:
    raise ValueError(
        "/health failed."
    )

model_info_response = client.get(
    "/model-info"
)

print(
    "Model info status:",
    model_info_response.status_code,
)
print(
    json.dumps(
        model_info_response.json(),
        indent=2,
    )
)

if model_info_response.status_code != 200:
    raise ValueError(
        "/model-info failed."
    )

## 8. Create one realistic request from the processed data

In [ ]:
reference_data = pd.read_csv(
    ML_DATA_FILE,
    parse_dates=["Date"],
)

reference_data = (
    reference_data
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

sample_row = (
    reference_data[
        reference_data["Ticker"] == "NVDA"
    ]
    .iloc[-1]
)

sample_payload = {
    feature: float(
        sample_row[feature]
    )
    for feature in FEATURE_COLUMNS
}

sample_payload["Ticker"] = str(
    sample_row["Ticker"]
)

print(
    json.dumps(
        sample_payload,
        indent=2,
    )
)

## 9. Call `/predict`

In [ ]:
prediction_response = client.post(
    "/predict",
    json=sample_payload,
)

print(
    "Status:",
    prediction_response.status_code,
)
print(
    json.dumps(
        prediction_response.json(),
        indent=2,
    )
)

if prediction_response.status_code != 200:
    raise ValueError(
        "/predict failed."
    )

## 10. Verify deployment parity

The direct model and API must return the same probability for the same row.

In [ ]:
direct_model = joblib.load(
    CHAMPION_MODEL_FILE
)

direct_input = pd.DataFrame(
    [sample_payload],
    columns=MODEL_COLUMNS,
)

direct_probability = float(
    direct_model
    .predict_proba(
        direct_input
    )[0, 1]
)

api_probability = float(
    prediction_response.json()[
        "large_move_probability"
    ]
)

print(
    "Direct probability:",
    direct_probability,
)
print(
    "API probability:",
    api_probability,
)

parity_passed = np.isclose(
    direct_probability,
    api_probability,
    rtol=1e-12,
    atol=1e-12,
)

print(
    "Parity passed:",
    parity_passed,
)

if not parity_passed:
    raise ValueError(
        "API prediction differs "
        "from direct model prediction."
    )

## 11. Test request validation

In [ ]:
invalid_ticker_payload = {
    **sample_payload,
    "Ticker": "AAPL",
}

invalid_ticker_response = (
    client.post(
        "/predict",
        json=invalid_ticker_payload,
    )
)

missing_feature_payload = (
    sample_payload.copy()
)
missing_feature_payload.pop(
    "Return_1D"
)

missing_feature_response = (
    client.post(
        "/predict",
        json=missing_feature_payload,
    )
)

extra_field_payload = {
    **sample_payload,
    "made_up_feature": 123,
}

extra_field_response = (
    client.post(
        "/predict",
        json=extra_field_payload,
    )
)

print(
    "Invalid ticker:",
    invalid_ticker_response.status_code,
)
print(
    "Missing feature:",
    missing_feature_response.status_code,
)
print(
    "Extra field:",
    extra_field_response.status_code,
)

expected_codes = [
    invalid_ticker_response.status_code,
    missing_feature_response.status_code,
    extra_field_response.status_code,
]

if expected_codes != [422, 422, 422]:
    raise ValueError(
        "Request validation did not "
        "behave as expected."
    )

print("Request validation tests passed.")

## 12. Save reusable pytest tests

In [ ]:
TEST_SOURCE = 'import sys\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom fastapi.testclient import TestClient\n\n\nAPI_DIR = Path(__file__).resolve().parents[1]\n\nif str(API_DIR) not in sys.path:\n    sys.path.insert(0, str(API_DIR))\n\nfrom app import app\nfrom model_service import (\n    FEATURE_COLUMNS,\n    MODEL_COLUMNS,\n    MODEL_FILE,\n)\n\n\nclient = TestClient(app)\n\n\ndef make_sample(ticker="NVDA"):\n    sample = {\n        feature: 0.0\n        for feature in FEATURE_COLUMNS\n    }\n\n    sample["Relative_Volume_20D"] = 1.0\n    sample["Ticker"] = ticker\n\n    return sample\n\n\ndef test_health():\n    response = client.get("/health")\n\n    assert response.status_code == 200\n    assert response.json()["status"] == "ok"\n    assert response.json()["model_loaded"] is True\n\n\ndef test_predict_matches_direct_model():\n    sample = make_sample()\n\n    response = client.post(\n        "/predict",\n        json=sample,\n    )\n\n    assert response.status_code == 200\n\n    model = joblib.load(MODEL_FILE)\n\n    direct_input = pd.DataFrame(\n        [sample],\n        columns=MODEL_COLUMNS,\n    )\n\n    direct_probability = float(\n        model.predict_proba(\n            direct_input\n        )[0, 1]\n    )\n\n    api_probability = float(\n        response.json()[\n            "large_move_probability"\n        ]\n    )\n\n    assert np.isclose(\n        direct_probability,\n        api_probability,\n        rtol=1e-12,\n        atol=1e-12,\n    )\n\n\ndef test_invalid_ticker_is_rejected():\n    response = client.post(\n        "/predict",\n        json=make_sample(\n            ticker="AAPL"\n        ),\n    )\n\n    assert response.status_code == 422\n'
TEST_DIR = API_DIR / 'tests'
TEST_DIR.mkdir(parents=True, exist_ok=True)
TEST_FILE = TEST_DIR / 'test_api.py'
TEST_FILE.write_text(TEST_SOURCE, encoding='utf-8')
print('Created:', TEST_FILE)

## 13. Final package verification

In [ ]:
expected_files = [
    API_DIR / "app.py",
    API_DIR / "schemas.py",
    API_DIR / "model_service.py",
    API_DIR / "requirements.txt",
        API_DIR / "artifacts" / "model_metadata.json",
    API_DIR
    / "artifacts"
    / "logistic_regression_candidate.joblib",
    API_DIR
    / "tests"
    / "test_api.py",
]

for file_path in expected_files:
    print(
        file_path.relative_to(
            PROJECT_ROOT
        ),
        "exists:",
        file_path.exists(),
    )

    if not file_path.exists():
        raise FileNotFoundError(
            file_path
        )

print()
print(
    "FastAPI inference pipeline "
    "completed successfully."
)
print(
    "API/direct-model prediction "
    "parity verified."
)

# What to send back

Send:

1. FastAPI, Pydantic and Uvicorn versions
2. `/health` response
3. `/model-info` response
4. `/predict` response
5. Direct probability versus API probability
6. The three validation-test status codes

Once these pass, we move directly to Docker.